***

*Course:* [Math 535](https://people.math.wisc.edu/~roch/mmids/) - Mathematical Methods in Data Science (MMiDS)  
*Chapter:* 2-Least squares   
*Author:* [Sebastien Roch](https://people.math.wisc.edu/~roch/), Department of Mathematics, University of Wisconsin-Madison  
*Updated:* Jan 30, 2024   
*Copyright:* &copy; 2024 Sebastien Roch

***

In [ ]:
# FOR e-BOOK ONLY
import os, sys
sys.path.insert(0, os.path.abspath('../../utils')) # use directory to mmids.py

In [ ]:
# PYTHON 3
import numpy as np
from numpy import linalg as LA
import matplotlib.pyplot as plt
import pandas as pd
import mmids
seed = 535
rng = np.random.default_rng(seed)
import warnings
warnings.filterwarnings('ignore')

In [ ]:
# FOR TeX ONLY
#plt.rcParams['figure.figsize'] = [4.00, 2.25] # for high-def figs
#plt.rcParams['figure.dpi'] = 200 # for high-def figs

## Background: review of vector spaces and matrix inverses

In this section, we introduce some basic linear algebra concepts that will be needed throughout this chapter (and later on).

### Subspaces

Throughout, we work over the vector space $V = \mathbb{R}^n$.  We will use the notation $[m] = \{1,\ldots,m\}$. 

We begin with the concept of a linear subspace.

**DEFINITION** **(Linear Subspace)** A linear subspace of $\mathbb{R}^n$ is a subset $U \subseteq \mathbb{R}^n$ that is closed under vector addition and scalar multiplication. That is, for all $\mathbf{u}_1, \mathbf{u}_2 \in U$ and $\alpha \in \mathbb{R}$, it holds that

$$
\mathbf{u}_1 + \mathbf{u}_2 \in U \quad \text{and} \quad \alpha \,\mathbf{u}_1 \in U.
$$

It follows from this condition that $\mathbf{0} \in U$. $\natural$

Alternatively, we can check these conditions by proving that (1) $\mathbf{0} \in U$ and (2) $\mathbf{u}_1, \mathbf{u}_2 \in U$ and $\alpha \in \mathbb{R}$ imply that $\alpha \mathbf{u}_1 + \mathbf{u}_2 \in U$. Indeed, taking $\alpha = 1$ gives the first condition above, while choosing $\mathbf{u}_2 = \mathbf{0}$ gives the second one. 

**NUMERICAL CORNER:** The plane $P$ made of all points $(x,y,z) \in \mathbb{R}^3$ that satisfy $z = x+y$ is a linear subspace. Indeed, $0 = 0 + 0$ so $(0,0,0) \in P$. And, for any $\mathbf{u}_1 = (x_1, y_1, z_1)$ and $\mathbf{u}_2 = (x_2, y_2, z_2)$ such that $z_1 = x_1 + y_1$ and $z_2 = x_2 + y_2$ and for any $\alpha \in \mathbb{R}$, we have

$$
\alpha z_1 + z_2 = \alpha (x_1 + y_1) + (x_2 + y_2) = (\alpha x_1 + x_2) + (\alpha y_1 + y_2).
$$

That is, $\alpha \mathbf{u}_1 + \mathbf{u}_2$ satisfies the condition defining $P$ and therefore is itself in $P$. Note also that $P$ passes through the origin.

In this example, the linear subspace $P$ can be described alternatively as the collection of every vector of the form $(x, y, x+y)$.

We use [`plot_surface`](https://matplotlib.org/stable/api/_as_gen/mpl_toolkits.mplot3d.axes3d.Axes3D.html#mpl_toolkits.mplot3d.axes3d.Axes3D.plot_surface) to plot it over a grid of points created using [`numpy.meshgrid`](https://numpy.org/doc/stable/reference/generated/numpy.meshgrid.html).

In [ ]:
x = np.linspace(0,1,num=101)
y = np.linspace(0,1,num=101)
X, Y = np.meshgrid(x, y)

In [ ]:
print(X)

In [ ]:
print(Y)

In [ ]:
Z = X + Y
print(Z)

In [ ]:
fig = plt.figure()
ax = fig.add_subplot(111, projection='3d')
ax.plot_surface(X, Y, Z)
plt.show()

$\unlhd$

Here is a key example of a linear subspace.

**DEFINITION** **(Span)** Let $\mathbf{w}_1, \ldots, \mathbf{w}_m \in \mathbb{R}^n$. The span of $\{\mathbf{w}_1, \ldots, \mathbf{w}_m\}$, denoted $\mathrm{span}(\mathbf{w}_1, \ldots, \mathbf{w}_m)$, is the set of all linear combinations of the $\mathbf{w}_j$'s. That is,

$$
\mathrm{span}(\mathbf{w}_1, \ldots, \mathbf{w}_m)
= \left\{
\sum_{j=1}^m \alpha_j \mathbf{w}_j\,:\, \alpha_1,\ldots, \alpha_m \in \mathbb{R}
\right\}.
$$

By convention, we declare that the span of the empty list is $\{\mathbf{0}\}$. $\natural$

**EXAMPLE:** **(continued)** In the example above, we noted that the plane $P$ is the collection of every vector of the form $(x, y, x+y)^T$. These can be written as $x \,\mathbf{w}_1 + y \,\mathbf{w}_2$ where $\mathbf{w}_1 = (1,0,1)^T$ and $\mathbf{w}_2 = (0,1,1)^T$, and vice versa. Hence $P = \mathrm{span}(\mathbf{w}_1,\mathbf{w}_2)$. $\lhd$

We check next that a span is indeed a linear subspace:

**LEMMA** Let $W=\mathrm{span}(\mathbf{w}_1, \ldots, \mathbf{w}_m)$. Then $W$ is a linear subspace. $\flat$

*Proof:* First, $\mathbf{0} = \sum_{j=1}^m 0\mathbf{w}_j \in W$. Second, let $\mathbf{u}_1, \mathbf{u}_2 \in W$ and $\alpha \in \mathbb{R}$. For $i=1,2$, because $\mathbf{u}_i$ is in the span of the $\mathbf{w}_j$'s, we can write

$$
\mathbf{u}_i = \sum_{j=1}^m \beta_{ij} \mathbf{w}_j
$$

for some $\beta_{ij} \in \mathbb{R}$ for $j=1,\ldots,m$.

Therefore

$$
\alpha \mathbf{u}_1 + \mathbf{u}_2 
= \alpha \sum_{j=1}^m \beta_{1j} \mathbf{w}_j
+ \sum_{j=1}^m \beta_{2j} \mathbf{w}_j
= \sum_{j=1}^m (\alpha \beta_{1j} + \beta_{2j}) \mathbf{w}_j.
$$

So $\alpha \,\mathbf{u}_1 + \mathbf{u}_2 \in W$.$\square$

In matrix form, we talk about the column space of a (not necessarily square) matrix.

**DEFINITION** **(Column Space)** Let $A \in \mathbb{R}^{n\times m}$ be an $n\times m$ matrix with columns $\mathbf{a}_1,\ldots, \mathbf{a}_m \in \mathbb{R}^n$. The column space of $A$, denoted $\mathrm{col}(A)$, is the span of the columns of $A$, that is, $\mathrm{col}(A) = \mathrm{span}(\mathbf{a}_1,\ldots, \mathbf{a}_m)$. $\natural$

(When thinking of $A$ as a linear map, the column space is often referred to as the range or image.)

We will need another important linear subspace defined in terms of a matrix. This one is defined implicitly in terms of the given matrix.

**DEFINITION** **(Null Space)** Let $B \in \mathbb{R}^{n \times m}$. The null space of $B$ is the linear subspace

$$
\mathrm{null}(B)
= \left\{\mathbf{x} \in \mathbb{R}^m\,:\, B\mathbf{x} = \mathbf{0}\right\}.
$$

$\natural$

It can be shown that the null space is a linear subspace. We give a simple example next.

**EXAMPLE:** **(continued)** Going back to the linear subspace $P = \{(x,y,z)^T \in \mathbb{R}^3 : z = x + y\}$,  the condition in the definition can be re-written as $x + y - z = 0$. Hence $P = \mathrm{null}(B)$ for the single-row matrix $B = \begin{pmatrix} 1 & 1 & - 1\end{pmatrix}$. $\lhd$

### Linear independence and rank

It is often desirable to avoid redundancy in the description of a linear subspace. 

We start with an example.

**EXAMPLE:** Consider the linear subspace $\mathrm{span}(\mathbf{w}_1,\mathbf{w}_2,\mathbf{w}_3)$, where $\mathbf{w}_1 = (1,0,1)^T$, $\mathbf{w}_2 = (0,1,1)^T$, and $\mathbf{w}_3 = (1,-1,0)^T$. We claim that 

$$
\mathrm{span}(\mathbf{w}_1,\mathbf{w}_2,\mathbf{w}_3) = \mathrm{span}(\mathbf{w}_1,\mathbf{w_2}).
$$

Recall that to prove an equality between sets, it suffices to prove inclusion in both directions. 

First, it is immediate by definition of the span that 

$$
\mathrm{span}(\mathbf{w}_1,\mathbf{w}_2) \subseteq \mathrm{span}(\mathbf{w}_1,\mathbf{w}_2,\mathbf{w}_3).
$$

To prove the other direction, let $\mathbf{u} 
\in \mathrm{span}(\mathbf{w}_1,\mathbf{w}_2,\mathbf{w}_3)$ so that

$$
\mathbf{u}
= \beta_1\,(1,0,1)^T + \beta_2\,(0,1,1)^T + \beta_3\,(1,-1,0)^T.
$$

Now observe that $(1,-1,0)^T = (1,0,1)^T - (0,1,1)^T$. Put differently, $\mathbf{w}_3 \in \mathrm{span}(\mathbf{w}_1,\mathbf{w}_2)$. Replacing above gives

\begin{align*}
\mathbf{u}
&= \beta_1\,(1,0,1)^T + \beta_2\,(0,1,1)^T + \beta_3\,[(1,0,1)^T - (0,1,1)^T]\\
&= (\beta_1+\beta_3)\,(1,0,1)^T + (\beta_2-\beta_3)\,(0,1,1)^T
\end{align*}

which shows that $\mathbf{u} \in \mathrm{span}(\mathbf{w}_1,\mathbf{w_2})$.
In words, $(1,-1,0)^T$ is redundant.
Hence 

$$
\mathrm{span}(\mathbf{w}_1,\mathbf{w}_2) \supseteq \mathrm{span}(\mathbf{w}_1,\mathbf{w}_2,\mathbf{w}_3)
$$ 

and that concludes the proof.$\lhd$

**DEFINITION** **(Linear Independence)** A list of nonzero vectors $\mathbf{u}_1,\ldots,\mathbf{u}_m$ is linearly independent if none of them can be written as a linear combination of the others, that is,

$$
\forall i,\ \mathbf{u}_i \notin \mathrm{span}(\{\mathbf{u}_j:j\neq i\}).
$$

By convention, we declare the empty list to be linearly independent. A list of vectors is called linearly dependent if it is not linearly independent. $\natural$

**EXAMPLE:** **(continued)** In the previous example, $\mathbf{w}_1,\mathbf{w}_2,\mathbf{w}_3$ are *not* linearly independent, because we showed that $\mathbf{w}_3$ can be written as a linear combination of $\mathbf{w}_1,\mathbf{w}_2$. On the other hand, $\mathbf{w}_1,\mathbf{w}_2$ are linearly independent because there is no $\alpha, \beta  \in \mathbb{R}$ such that $(1,0,1)^T = \alpha\,(0,1,1)^T$ or $(0,1,1)^T = \beta\,(1,0,1)^T$. Indeed, the first equation requires $1 = \alpha \, 0$ (first component) and the second one requires $1 = \beta \, 0$ (second component) - both of which have no solution.
$\lhd$

**LEMMA** **(Equivalent Definition of Linear Independence)** Vectors $\mathbf{u}_1,\ldots,\mathbf{u}_m$ are linearly independent if and only if

$$
\sum_{j=1}^m \alpha_j \mathbf{u}_j = \mathbf{0} \implies \alpha_j = 0,\ \forall j.
$$

Equivalently, $\mathbf{u}_1,\ldots,\mathbf{u}_m$ are linearly dependent if and only if there exist $\alpha_j$'s, not all zero, such that $\sum_{j=1}^m \alpha_j \mathbf{u}_j = \mathbf{0}$. 

$\flat$

*Proof:* We prove the second statement.

- Assume $\mathbf{u}_1,\ldots,\mathbf{u}_m$ are linearly dependent. Then $\mathbf{u}_i = \sum_{j\neq i} \alpha_j \mathbf{u}_j$ for some $i$. Taking $\alpha_i = -1$ gives $\sum_{j=1}^m \alpha_j \mathbf{u}_j = \mathbf{0}$.

- Assume $\sum_{j=1}^m \alpha_j \mathbf{u}_j = \mathbf{0}$ with $\alpha_j$'s not all zero. In particular $\alpha_i \neq 0$ for some $i$. Then 

$$
\mathbf{u}_i 
= - \frac{1}{\alpha_i} \sum_{j\neq i} \alpha_j \mathbf{u}_j
= \sum_{j\neq i} \left(- \frac{\alpha_j}{\alpha_i}\right) \mathbf{u}_j.
$$

$\square$ 

In matrix form: let $\mathbf{a}_1,\ldots,\mathbf{a}_m \in \mathbb{R}^n$ and form the matrix whose columns are the $\mathbf{a}_i$'s

$$
A =
\begin{pmatrix}
| &  & | \\
\mathbf{a}_1 & \ldots & \mathbf{a}_m \\
| &  & | 
\end{pmatrix}.
$$

Note that $A\mathbf{x}$ is the following linear combination of the columns of $A$: $\sum_{j=1}^m x_j \mathbf{a}_j$. Hence $\mathbf{a}_1,\ldots,\mathbf{a}_m$ are linearly independent if and only if
$A \mathbf{x} = \mathbf{0} \implies \mathbf{x} = \mathbf{0}$. In terms of the null space of $A$, this last condition translates into $\mathrm{null}(A) = \{\mathbf{0}\}$.

Equivalently, $\mathbf{a}_1,\ldots,\mathbf{a}_m$ are linearly dependent if and only if $\exists \mathbf{x}\neq \mathbf{0}$ such that $A \mathbf{x} = \mathbf{0}$. Put differently, this last condition means that there is a nonzero vector in the null space of $A$.

In this course, we will *not* be interested in checking these types of conditions by hand (although you may have learned to do so in a prior course).

Bases give a convenient - non-redundant - representation of a subspace.

**DEFINITION** **(Basis)** Let $U$ be a linear subspace of $\mathbb{R}^n$. A basis of $U$ is a list of vectors $\mathbf{u}_1,\ldots,\mathbf{u}_m$ in $U$ that: (1) span $U$, that is, $U = \mathrm{span}(\mathbf{u}_1,\ldots,\mathbf{u}_m)$; and (2) are linearly independent. $\natural$

We denote by $\mathbf{e}_1, \ldots, \mathbf{e}_n$ the standard basis of $\mathbb{R}^n$, where $\mathbf{e}_i$ has a one in coordinate $i$ and zeros in all other coordinates (see the example below). The basis of the linear subspace $\{\mathbf{0}\}$ is the empty list (which, by convention, is independent and has span $\{\mathbf{0}\}$).

**EXAMPLE:** **(continued)** In the previous example, the vectors $\mathbf{w}_1,\mathbf{w}_2$ are a basis of their span $U = \mathrm{span}(\mathbf{w}_1,\mathbf{w}_2)$. Indeed the first condition is trivially satisfied. Plus, we have shown above that $\mathbf{w}_1,\mathbf{w}_2$ are linearly independent. $\lhd$

**EXAMPLE:** For $i=1,\ldots,n$, recall that $\mathbf{e}_i \in \mathbb{R}^n$ is the vector with entries

$$
(\mathbf{e}_i)_j 
= \begin{cases}
1, & \text{if $j = i$,}\\
0, & \text{o.w.}
\end{cases}
$$

Then $\mathbf{e}_1, \ldots, \mathbf{e}_n$ form a basis of $\mathbb{R}^n$, as each vector is in $\mathbb{R}^n$. It is known as the standard basis of $\mathbb{R}^n$. Indeed, clearly $\mathrm{span}(\mathbf{e}_1, \ldots, \mathbf{e}_n) \subseteq \mathbb{R}^n$. Moreover, any vector $\mathbf{u} = (u_1,\ldots,u_n)^T \in \mathbb{R}^n$ can be written as

$$
\mathbf{u}
= \sum_{i=1}^{n} u_i \mathbf{e}_i.
$$

So $\mathbf{e}_1, \ldots, \mathbf{e}_n$ spans $\mathbb{R}^n$. Furthermore, 

$$
\mathbf{e}_i \notin \mathrm{span}(\{\mathbf{e}_j:j\neq i\}), \quad \forall i=1,\ldots,n,
$$

since $\mathbf{e}_i$ has a non-zero $i$-th entry while all vectors on the right-hand side have a zero in entry $i$. So the vectors $\mathbf{e}_1, \ldots, \mathbf{e}_n$ are linearly independent. $\lhd$

A key property of a basis is that it provides a *unique* representation of the vectors in the subspace. Indeed, let $U$ be a linear subspace and $\mathbf{u}_1,\ldots,\mathbf{u}_m$ be a basis of $U$. Suppose that $\mathbf{w} \in U$ can be written as both $\mathbf{w} = \sum_{j=1}^m \alpha_j \mathbf{u}_j$ and $\mathbf{w} = \sum_{j=1}^m \alpha_j' \mathbf{u}_j$. Then subtracting one equation from the other we arrive at $\sum_{j=1}^m (\alpha_j - \alpha_j') \,\mathbf{u}_j = \mathbf{0}$. By linear independence, we have $\alpha_j - \alpha_j' = 0$ for each $j$. That is, there is only one way to express $\mathbf{w}$ as a linear combination of the basis. 

The basis itself on the other hand is not unique. 

A second key property of a basis is that it always has the *same number of elements*, which is called the dimension of the subspace.

**THEOREM** **(Dimension)** Let $U \neq \{\mathbf{0}\}$ be a linear subspace of $\mathbb{R}^n$. Then all bases of $U$ have the same number of elements. We call this number the dimension of $U$ and denote it by $\mathrm{dim}(U)$. $\sharp$

The proof is optional and is provided in a later section. It relies on the *Linear Dependence Lemma*, which is also stated in the next section. That fundamental lemma has many useful implications, some of which we state now.

 A list of linearly independent vectors in a subspace $U$ is referred to as an independent list in $U$. A list of vectors whose span is $U$ is referred to as a spanning list of $U$. In the lemmas below, $U$ is a linear subspace of $\mathbb{R}^n$. The first and second lemmas are proved in a later section. The *Dimension Theorem* immediately follows from the first one (Why?).

**LEMMA** **(Independent is Shorter than Spanning)**  The length of any independent list in $U$ is less or equal than the length of any spanning list of $U$. $\flat$


**LEMMA** **(Completing an Independent List)** Any independent list in $U$ can be completed into a basis of $U$. $\flat$

**LEMMA** **(Reducing a Spanning List)** Any spanning list of $U$ can be reduced into a basis of $U$. $\flat$

We mention a few observations implied by the previous lemmas. 

**Observation D1:** Any linear subspace $U$ of $\mathbb{R}^n$ has a basis. To show this, start with the empty list and use the *Completing an Independent List Lemma* to complete it into a basis of $U$. Observe further that, instead of an empty list, we could have initialized the process with a list containing any vector in $U$. That is, for any non-zero vector $\mathbf{u} \in U$, we can construct a basis of $U$ that includes $\mathbf{u}$.

**Observation D2:** The dimension of any linear subspace $U$ of $\mathbb{R}^n$ is smaller or equal than $n$. Indeed, because a basis of $U$ is an independent list in the full space $\mathbb{R}^n$, by the *Completing an Independent List Lemma* it can be completed into a basis of $\mathbb{R}^n$, which has $n$ elements by *Dimension Theorem* (and the fact that the standard basis has $n$ elements). A similar statement holds more generally for nested linear subspaces $U \subseteq V$, that is, $\mathrm{dim}(U) \leq \mathrm{dim}(V)$.

When applied to a matrix $A$, the dimension of the column space of $A$ is called the column rank of $A$. Similarly the row rank of $A$ is the dimension of its row space.

**DEFINITION** **(Row Space)** The row space of $A \in \mathbb{R}^{n \times m}$, denoted $\mathrm{row}(A)$, is the span of the rows of $A$ as vectors in $\mathbb{R}^m$. $\natural$

Observe that the row space of $A$ is equal to the column space of its transpose $A^T$.

As it turns out, these two notions of rank are the same. From now on, we refer to the row rank and column rank of $A$ simply as the rank, which we denote by $\mathrm{rk}(A)$.

**THEOREM** **(Row Rank Equals Column Rank)** For any $A \in \mathbb{R}^{n \times m}$, the row rank of $A$ equals the column rank of $A$. Moreover, $\mathrm{rk}(A) \leq \min\{n,m\}$. $\sharp$

*Proof idea:* Write $A$ as a matrix factorization $BC$ where the columns of $B$ form a basis of $\mathrm{col}(A)$. Then the rows of $C$ necessarily form a spanning set of $\mathrm{row}(A)$. So, because the number of columns of $B$ and the number of rows of $C$ match, we conclude that the row rank is less or equal than the column rank. Applying the same argument to the transpose gives the claim.

*Proof:* Assume that $A$ has column rank $r$. Then there exists a basis $\mathbf{b}_1,\ldots, \mathbf{b}_r \in \mathbb{R}^n$ of $\mathrm{col}(A)$ by *Observation D1* above, and we know that $r \leq n$ by *Observation D2*. That is, for each $j$, letting $\mathbf{a}_{j} = A_{\cdot,j}$ be the $j$-th column of $A$ we can write

$$
\mathbf{a}_{j} = \sum_{\ell=1}^r \mathbf{b}_\ell c_{\ell j}
$$

for some $c_{\ell j}$'s. Let $B$ be the matrix whose columns are $\mathbf{b}_1, \ldots, \mathbf{b}_r$ and let $C$ be the matrix with entries $(C)_{\ell j} = c_{\ell j}$, $\ell=1,\ldots,r$, $j=1,\ldots,m$. Then the equation above can be re-written as the matrix factorization $A = BC$. Indeed, by our previous observations about matrix-matrix products, the columns of $A$ are linear combinations of the columns of $B$ with coefficients taken from the corresponding column of $C$.

The key point is the following: $C$ necessarily has $r$ rows. Let $\boldsymbol{\alpha}_{i}^T = A_{i,\cdot}$ be the $i$-th row of $A$ and $\mathbf{c}_{\ell}^T = C_{\ell,\cdot}$ be the $\ell$-th row of $C$. Using our alternative representation of matrix-matrix product in terms of rows, the decomposition is equivalent to

$$
\boldsymbol{\alpha}_{i}^T = \sum_{\ell=1}^r b_{i\ell} \mathbf{c}_\ell^T, \quad i=1,\ldots, n,
$$

where $b_{i\ell} = (\mathbf{b}_i)_\ell = (B)_{i\ell}$ is the $i$-th entry of the $\ell$-th column of $B$. In words, the rows of $A$ are linear combinations of the rows of $C$ with coefficients taken from the corresponding row of $B$. In particular, $\mathcal{C} = \{\mathbf{c}_{j}:j=1,\ldots,r\}$ is a spanning list of the row space of $A$, that is, each row of $A$ can be written as a linear combination of $\mathcal{C}$. 

So the row rank of $A$ is at most $r$, the column rank of $A$, by *Observation D2*.

Applying the same argument to $A^T$, which switches the role of the columns and the rows, gives that the column rank of $A$ (i.e. the row rank of $A^T$) is at most the row rank of $A$ (i.e. the column rank of $A^T$). Hence the two notions of rank must be equal. (We also deduce $r \leq m$ by *Observation D2* again.) $\square$

**EXAMPLE:** **(continued)** We illustrate the proof of the theorem. Continuing a previous example, let $A$ be the matrix with columns $\mathbf{w}_1 = (1,0,1)^T$, $\mathbf{w}_2 = (0,1,1)^T$, and $\mathbf{w}_3 = (1,-1,0)^T$

$$
A
= 
\begin{pmatrix}
1 & 0 & 1\\
0 & 1 & -1\\
1 & 1 & 0
\end{pmatrix}.
$$

We know that $\mathbf{w}_1$ and $\mathbf{w}_2$ form a basis of $\mathrm{col}(A)$. We use them to construct our matrix $B$

$$
B
= 
\begin{pmatrix}
1 & 0\\
0 & 1\\
1 & 1
\end{pmatrix}.
$$

Recall that $\mathbf{w}_3 = \mathbf{w}_1 - \mathbf{w}_2$. Hence the matrix $C$ is

$$
C
= 
\begin{pmatrix}
1 & 0 & 1\\
0 & 1 & -1
\end{pmatrix}.
$$

Indeed, column $j$ of $C$ gives the coefficients in the linear combination of the columns of $B$ that produces column $j$ of A. Check that $A = B C$. $\lhd$

**NUMERICAL CORNER:** In Numpy, one can compute the rank of a matrix using the function [`numpy.linalg.matrix_rank`](https://numpy.org/doc/stable/reference/generated/numpy.linalg.matrix_rank.html). We will see later in the course how to compute it using the singular value decomposition (which is how `LA.matrix_rank` does it).

Let's try the example above.

In [ ]:
w1 = np.array([1., 0., 1.])
w2 = np.array([0., 1., 1.])
w3 = np.array([1., -1., 0.])
A = np.stack((w1, w2, w3), axis=-1)
print(A)

We compute the rank of `A`.

In [ ]:
LA.matrix_rank(A)

We take only the first two columns of `A` this time to form `B`.

In [ ]:
B = np.stack((w1, w2),axis=-1)
print(B)

In [ ]:
LA.matrix_rank(B)

In Numpy, `@` is used for matrix product.

In [ ]:
C = np.array([[1., 0., 1.],[0., 1., -1.]])
print(C)

In [ ]:
LA.matrix_rank(C)

In [ ]:
print(B @ C)

$\unlhd$

**EXAMPLE:** Let $A \in \mathbb{R}^{n \times k}$ and $B \in \mathbb{R}^{k \times m}$. Then we claim that

$$
\mathrm{rk}(AB) \leq \mathrm{rk}(A).
$$

Indeed, the columns of $AB$ are linear combinations of the columns of $A$. Hence $\mathrm{col}(AB) \subseteq \mathrm{col}(A)$. Because a basis of $\mathrm{col}(AB)$ is an independent list in $\mathrm{col}(A)$, it can be completed into a basis of $\mathrm{col}(A)$ by the *Completing an Independent List Lemma*. The length of the resulting basis is greater or equal than the length of the basis of $\mathrm{col}(AB)$. Because the rank of a matrix is the length of a basis of its column space, the claim follows. $\lhd$

**EXAMPLE:** Let $A \in \mathbb{R}^{n \times k}$ and $B \in \mathbb{R}^{k \times m}$. Then we claim that

$$
\mathrm{rk}(A + B) \leq \mathrm{rk}(A) + \mathrm{rk}(B).
$$

Indeed, the columns of $A + B$ are linear combinations of the columns of $A$ and of $B$. Let $\mathbf{u}_1,\ldots,\mathbf{u}_h$ be a basis of $\mathrm{col}(A)$ and let $\mathbf{v}_1,\ldots,\mathbf{v}_{\ell}$ be a basis of $\mathrm{col}(B)$. Then, we deduce 

$$
\mathrm{col}(A + B) \subseteq \mathrm{span}(\mathbf{u}_1,\ldots,\mathbf{u}_h,\mathbf{v}_1,\ldots,\mathbf{v}_{\ell}).
$$ 

Using the *Completing an Independent List Lemma* as in the previous example, it follows that

$$
\mathrm{rk}(A + B) \leq \mathrm{dim}(\mathrm{span}(\mathbf{u}_1,\ldots,\mathbf{u}_h,\mathbf{v}_1,\ldots,\mathbf{v}_{\ell})).
$$ 

By the *Reducing a Spanning List Lemma*, the right hand side is at most the length of the spanning list, i.e., $h+\ell$. But by construction $\mathrm{rk}(A) = h$ and $\mathrm{rk}(B) = \ell$, so we are done. $\lhd$

### A key concept: orthogonality

Orthogonality plays a key role in linear algebra for data science thanks to its computational properties and its connection to the least-squares problem.

**DEFINITION** **(Orthogonality)** Vectors $\mathbf{u}$ and $\mathbf{v}$ in $\mathbb{R}^n$ (as column vectors) are orthogonal if their inner product is zero 

$$
\langle \mathbf{u}, \mathbf{v} \rangle 
=\mathbf{u}^T \mathbf{v}
= \sum_{i=1}^n u_i v_i 
= 0.
$$

$\natural$

Throughout this section, we will need a few standard properties of inner products. For one, they are symmetric in the sense that

$$
\langle \mathbf{x}, \mathbf{y} \rangle 
= \langle \mathbf{y}, \mathbf{x} \rangle \qquad \forall \mathbf{x}, \mathbf{y} \in \mathbb{R}^n.
$$

Second, they are linear in each input: for any $\mathbf{x}_1, \mathbf{x}_2, \mathbf{x}_3 \in \mathbb{R}^n$ and $\beta \in \mathbb{R}$, it holds that

$$
\langle \beta \,\mathbf{x}_1 + \mathbf{x}_2, \mathbf{x}_3 \rangle = \beta \,\langle \mathbf{x}_1,\mathbf{x}_3\rangle + \langle \mathbf{x}_2,\mathbf{x}_3\rangle.
$$

Repeated application of the latter property implies for instance that: for any $\mathbf{x}_1, \ldots, \mathbf{x}_m, \mathbf{y}_1, \ldots, \mathbf{y}_\ell, \in \mathbb{R}^n$,

$$
\left\langle \sum_{i=1}^m \mathbf{x}_i, \sum_{j=1}^\ell \mathbf{y}_j \right\rangle
= \sum_{i=1}^m \sum_{j=1}^\ell \langle \mathbf{x}_i,\mathbf{y}_j\rangle.
$$

Finally recall that $\|\mathbf{x}\|^2 = \langle \mathbf{x}, \mathbf{x} \rangle$. 

Orthogonality has important implications. The following classical result will be useful below. Throughout, we use $\|\mathbf{u}\|$ for the Euclidean norm of $\mathbf{u}$.

**LEMMA** **(Pythagoras)** Let $\mathbf{u}, \mathbf{v} \in \mathbb{R}^n$ be orthogonal. Then
$\|\mathbf{u} + \mathbf{v}\|^2 = \|\mathbf{u}\|^2 + \|\mathbf{v}\|^2$. $\flat$

*Proof:* Using $\|\mathbf{w}\|^2 = \langle \mathbf{w}, \mathbf{w}\rangle$, we get

\begin{align*}
\|\mathbf{u} + \mathbf{v}\|^2
&= \langle \mathbf{u} + \mathbf{v}, \mathbf{u} + \mathbf{v}\rangle\\
&= \langle \mathbf{u}, \mathbf{u}\rangle
+ 2 \,\langle \mathbf{u}, \mathbf{v}\rangle
+ \langle \mathbf{v}, \mathbf{v}\rangle\\
&= \|\mathbf{u}\|^2 + \|\mathbf{v}\|^2.
\end{align*}

$\square$

Here is an application of *Pythagoras*.

**LEMMA** **(Cauchy-Schwarz)** For any $\mathbf{u}, \mathbf{v} \in \mathbb{R}^n$,
$|\langle \mathbf{u}, \mathbf{v}\rangle| \leq \|\mathbf{u}\| \|\mathbf{v}\|$. $\flat$

*Proof:* Let $\mathbf{q} = \frac{\mathbf{v}}{\|\mathbf{v}\|}$ be the unit vector in the direction of $\mathbf{v}$. We want to show $|\langle \mathbf{u}, \mathbf{q}\rangle| \leq \|\mathbf{u}\|$. Decompose $\mathbf{u}$ as follows: 

$$
\mathbf{u} 
= \left\langle \mathbf{u}, \mathbf{q}\right\rangle \mathbf{q}
+ \left\{\mathbf{u} - \left\langle \mathbf{u}, \mathbf{q}\right\rangle \mathbf{q}\right\}.
$$

The two terms on the right-hand side are orthogonal: 

$$
\left\langle
\left\langle \mathbf{u}, \mathbf{q}\right\rangle \mathbf{q},
\mathbf{u} - \left\langle \mathbf{u}, \mathbf{q}\right\rangle \mathbf{q}
\right\rangle
= \left\langle \mathbf{u}, \mathbf{q}\right\rangle^2  - 
\left\langle \mathbf{u}, \mathbf{q}\right\rangle^2 \left\langle \mathbf{q}, \mathbf{q}\right\rangle
= 0.
$$

So *Pythagoras* gives

$$
\|\mathbf{u}\|^2
= \left\|\left\langle \mathbf{u}, \mathbf{q}\right\rangle \mathbf{q}\right\|^2
+ \left\|\mathbf{u} - \left\langle \mathbf{u}, \mathbf{q}\right\rangle \mathbf{q}\right\|^2
\geq \left\|\left\langle \mathbf{u}, \mathbf{q}\right\rangle \mathbf{q}\right\|^2
= \left\langle \mathbf{u}, \mathbf{q}\right\rangle^2.
$$

Taking a square root gives the claim. $\square$

**Orthonormal basis expansion** To begin to see the power of orthogonality, consider the following. A list of vectors $\{\mathbf{u}_1,\ldots,\mathbf{u}_m\}$ is an orthonormal list if the $\mathbf{u}_i$'s are pairwise orthogonal and each has norm 1, that is, for all $i$ and all $j \neq i$, we have $\|\mathbf{u}_i\| = 1$ and $\langle \mathbf{u}_i, \mathbf{u}_j \rangle = 0$. Alternatively,

$$
\langle \mathbf{u}_i, \mathbf{u}_j \rangle
= \begin{cases}
1 & \text{if $i=j$}\\
0 & \text{if $i\neq j$}.
\end{cases}
$$

**LEMMA** **(Properties of Orthonormal Lists)** Let $\{\mathbf{u}_1,\ldots,\mathbf{u}_m\}$ be an orthonormal list. Then:
1. for any $\alpha_j \in \mathbb{R}$, $j=1,\ldots,m$, we have

$$
\left\|\sum_{j=1}^m \alpha_j \mathbf{u}_j\right\|^2 = \sum_{j=1}^m \alpha_j^2,
$$

2. the vectors $\{\mathbf{u}_1,\ldots,\mathbf{u}_m\}$ are linearly independent.

$\flat$


*Proof:* For 1., using that $\|\mathbf{x}\|^2 = \langle \mathbf{x}, \mathbf{x} \rangle$, we have

\begin{align*}
\left\|\sum_{j=1}^m \alpha_j \mathbf{u}_j\right\|^2
&= \left\langle
\sum_{i=1}^m \alpha_i \mathbf{u}_i, \sum_{j=1}^m \alpha_j \mathbf{u}_j
\right\rangle\\
&= \sum_{i=1}^m \alpha_i \left\langle
 \mathbf{u}_i, \sum_{j=1}^m \alpha_j \mathbf{u}_j
\right\rangle\\
&= \sum_{i=1}^m \sum_{j=1}^m \alpha_i \alpha_j \left\langle
 \mathbf{u}_i,  \mathbf{u}_j
\right\rangle\\
&= \sum_{i=1}^m \alpha_i^2
\end{align*}

where we used orthonormality in the last equation, that is, $\langle
 \mathbf{u}_i,  \mathbf{u}_j \rangle$ is $1$ if $i=j$ and $0$ otherwise.

For 2., suppose $\sum_{i=1}^m \beta_i \mathbf{u}_i = \mathbf{0}$, then we must have by 1. that $\sum_{i=1}^m \beta_i^2 = 0$. That implies $\beta_i = 0$ for all $i$. Hence the $\mathbf{u}_i$'s are linearly independent. $\square$

Given a basis $\{\mathbf{u}_1,\ldots,\mathbf{u}_m\}$ of $U$, we know that: for any $\mathbf{w} \in U$, $\mathbf{w} = \sum_{i=1}^m \alpha_i \mathbf{u}_i$ for some $\alpha_i$'s. It is not immediately obvious in general how to find the $\alpha_i$'s. In the orthonormal case, however, there is a formula. We say that the basis $\{\mathbf{u}_1,\ldots,\mathbf{u}_m\}$ is orthonormal if it forms an orthonormal list.

**THEOREM** **(Orthonormal Expansion)** Let $\mathbf{q}_1,\ldots,\mathbf{q}_m$ be an orthonormal basis of $U$ and let $\mathbf{w} \in U$. Then

$$
\mathbf{w}
= \sum_{j=1}^m \langle \mathbf{w}, \mathbf{q}_j\rangle \,\mathbf{q}_j.
$$

$\sharp$

*Proof:* Because $\mathbf{w} \in U$, $\mathbf{w} = \sum_{i=1}^m \alpha_i \mathbf{q}_i$ for some $\alpha_i$. Take the inner product with $\mathbf{q}_j$ and use orthonormality:

$$
\langle \mathbf{w}, \mathbf{q}_j\rangle 
= \left\langle \sum_{i=1}^m \alpha_i \mathbf{q}_i, \mathbf{q}_j\right\rangle 
= \sum_{i=1}^m \alpha_i \langle \mathbf{q}_i, \mathbf{q}_j\rangle 
= \alpha_j.
$$

Hence, we have determined all $\alpha_j$'s in the basis expansion of $\mathbf{w}$. $\square$

**EXAMPLE:** Consider again the linear subspace $W = \mathrm{span}(\mathbf{w}_1,\mathbf{w}_2,\mathbf{w}_3)$, where $\mathbf{w}_1 = (1,0,1)^T$, $\mathbf{w}_2 = (0,1,1)^T$, and $\mathbf{w}_3 = (1,-1,0)^T$. We have shown that in fact 

$$
\mathrm{span}(\mathbf{w}_1,\mathbf{w}_2,\mathbf{w}_3) = \mathrm{span}(\mathbf{w}_1,\mathbf{w}_2).
$$

as $\mathbf{w}_1,\mathbf{w}_2$ form a basis of $W$. On the other hand, 

$$
\langle\mathbf{w}_1,\mathbf{w}_2\rangle
= (1)(0) + (0)(1) + (1)(1) = 0 + 0 + 1 = 1 \neq 0
$$

so this basis is not orthonormal. Indeed, an orthonormal list is necessarily an independent list, but the opposite may not hold.

To produce an orthonormal basis of $W$, we can first proceed by normalizing $\mathbf{w}_1$

$$
\mathbf{q}_1 
= \frac{\mathbf{w}_1}{\|\mathbf{w}_1\|}
= \frac{\mathbf{w}_1}{\sqrt{1^2 + 0^2 + 1^2}}
= \frac{1}{\sqrt{2}} \mathbf{w}_1.
$$

Then $\|\mathbf{q}_1\| = 1$ since, in general, by homogeneity of the norm

$$
\left\|\frac{\mathbf{w}_1}{\|\mathbf{w}_1\|}\right\|
= \frac{1}{\|\mathbf{w}_1\|} \|\mathbf{w}_1\| = 1.
$$

We then seek a second basis vector. It must satisfy two conditions in this case: 

- it must be of unit norm and be orthogonal to $\mathbf{q}_1$; and 
- $\mathbf{w}_2$ must be a linear combination of $\mathbf{q}_1$ and $\mathbf{q}_2$. 

The latter condition guarantees that $\mathrm{span}(\mathbf{q}_1,\mathbf{q}_2) = \mathrm{span}(\mathbf{w}_1,\mathbf{w}_2)$. (Formally, that would imply only that $\mathrm{span}(\mathbf{w}_1,\mathbf{w}_2) \subseteq \mathrm{span}(\mathbf{q}_1,\mathbf{q}_2)$. In this case, it is easy to see that the containment must go in the opposite direction as well. Why?)

The first condition translates into

$$
1 = \|\mathbf{q}_2\|^2
= q_{21}^2 + q_{22}^2 + q_{23}^2,
$$

where $\mathbf{q}_2 = (q_{21},q_{22},q_{23})^T$, and 

$$
0 = \langle\mathbf{q}_1, \mathbf{q}_2\rangle = \frac{1}{\sqrt{2}}\left[1\cdot q_{21} + 0 \cdot q_{22} + 1 \cdot q_{23}\right] = \frac{1}{\sqrt{2}}\left[q_{21} + q_{23}\right].
$$

That is, simplifying the second display and plugging into the first, $q_{23} = -q_{21}$ and $q_{22} = \sqrt{1 - 2 q_{21}^2}$.

The second condition translates into: there is $\beta_1, \beta_2 \in \mathbb{R}$ such that

$$
\mathbf{w}_2 
= \begin{pmatrix}
0 \\
1 \\
1
\end{pmatrix}
=
\beta_1 \mathbf{q}_1 + \beta_2 \mathbf{q}_2
= \beta_1 
\frac{1}{\sqrt{2}}
\begin{pmatrix}
1 \\
0 \\
1
\end{pmatrix}
+
\beta_2
\begin{pmatrix}
q_{21} \\
\sqrt{1-2 q_{21}^2} \\
-q_{21}
\end{pmatrix}.
$$

The first entry gives $\beta_1/\sqrt{2} + \beta_2 q_{21} = 0$ while the third entry gives
$\beta_1/\sqrt{2} - \beta_2 q_{21} = 1$. Adding up the equations gives $\beta_1 = 1/\sqrt{2}$. Plugging back into the first one gives $\beta_2 = -1/(2q_{21})$. Returning to the equation for $\mathbf{w}_2$, we get from the second entry

$$
1 = - \frac{1}{2 q_{21}} \sqrt{1 - 2 q_{21}^2}.
$$

Rearranging and taking a square, we want the negative solution to

$$
4 q_{21}^2 = 1 - 2 q_{21}^2,
$$

that is, $q_{21} = - 1/\sqrt{6}$. Finally, we get $q_{23} = - q_{21} = 1/\sqrt{6}$ and 
$q_{22} = \sqrt{1 - 2 q_{21}^2} = \sqrt{1 - 1/3} = \sqrt{2/3} = 2/\sqrt{6}$.

To summarize, we have 

$$
\mathbf{q}_1
= \frac{1}{\sqrt{2}} \begin{pmatrix}
1\\
0\\
1
\end{pmatrix},
\quad
\mathbf{q}_2
= \frac{1}{\sqrt{6}} \begin{pmatrix}
-1\\
2\\
1
\end{pmatrix}.
$$

We confirm that 

$$
\langle \mathbf{q}_1, \mathbf{q}_2\rangle
= \frac{1}{\sqrt{2}\sqrt{6}}[(1)(-1) + (0)(2) + (1)(1)]
= 0
$$

and

$$
\|\mathbf{q}_2\|^2
= \left(-\frac{1}{\sqrt{6}}\right)^2 
+ \left(\frac{2}{\sqrt{6}}\right)^2 
+ \left(\frac{1}{\sqrt{6}}\right)^2
= \frac{1}{6} + \frac{4}{6} + \frac{1}{6}
= 1.
$$

We can use the *Orthonormal Expansion* to write $\mathbf{w}_2$ as a linear combination of $\mathbf{q}_1$ and $\mathbf{q}_2$. The inner products are

$$
\langle
\mathbf{w}_2, \mathbf{q}_1
\rangle
= 0 \left(\frac{1}{\sqrt{2}}\right)
+ 1 \left(\frac{0}{\sqrt{2}}\right)
+ 1 \left(\frac{1}{\sqrt{2}}\right)
= \frac{1}{\sqrt{2}},
$$

$$
\langle
\mathbf{w}_2, \mathbf{q}_2
\rangle
= 0 \left(-\frac{1}{\sqrt{6}}\right)
+ 1 \left(\frac{2}{\sqrt{6}}\right)
+ 1 \left(\frac{1}{\sqrt{6}}\right)
= \frac{3}{\sqrt{6}}.
$$

So

$$
\mathbf{w}_2
= \frac{1}{\sqrt{2}} \mathbf{q}_1 + \frac{3}{\sqrt{6}} \mathbf{q}_2.
$$

Check it! Try $\mathbf{w}_3$. $\lhd$

**Gram-Schmidt** We have shown that working with orthonormal bases is desirable. What if we do not have one? We could try to construct one by hand as we did in the previous example. But there are better ways. We review the [Gram-Schmidt algorithm](https://en.wikipedia.org/wiki/Gram–Schmidt_process) in an upcoming section, which will imply that every linear subspace has an orthonormal basis. That is, we will prove the following theorem.

**THEOREM** **(Gram-Schmidt)** Let $\mathbf{a}_1,\ldots,\mathbf{a}_m$ be linearly independent. Then there exists an orthonormal basis $\mathbf{q}_1,\ldots,\mathbf{q}_m$ of 
$\mathrm{span}(\mathbf{a}_1,\ldots,\mathbf{a}_m)$. $\sharp$

But, first, we define the orthogonal projection, which will play a key role in our applications. 

### Inverses

Recall the following important definition.

**DEFINITION** **(Nonsingular Matrix)** A square matrix $A \in \mathbb{R}^{n \times n}$ is nonsingular if it has rank $n$. In that case, we also say that $A$ is of full rank. $\natural$

An implication of this is that $A$ is nonsingular if and only if its columns form a basis of $\mathbb{R}^n$. Indeed, suppose the columns of $A$ form a basis of $\mathbb{R}^n$. Then the dimension of $\mathrm{col}(A)$ is $n$, which means the rank is $n$. In the other direction, suppose $A$ has rank $n$. 

1. We first prove a general statement: the columns of $Z \in \mathbb{R}^{k \times m}$ form a basis of $\mathrm{col}(Z)$ whenever $Z$ is of full column rank. Indeed, the columns of $Z$ by definition span $\mathrm{col}(Z)$. By the *Reducing a Spanning List Lemma*, they can be reduced into a basis of $\mathrm{col}(Z)$. If $Z$ has full column rank, then the length of any basis of $\mathrm{col}(Z)$ is equal to the number of columns of $Z$. So the columns of $Z$ must already form a basis.

2. Apply the previous claim to $Z = A$. Then, since the columns of $A$ form an independent list in $\mathbb{R}^n$, by the *Completing an Independent List Lemma* they can be completed into a basis of $\mathbb{R}^n$. But there are already $n$ of them, the dimension of $\mathbb{R}^n$, so they must already form a basis of $\mathbb{R}^n$. In other words, we have proved another general fact: an independent list of length $n$ in $\mathbb{R}^n$ is a basis of $\mathbb{R}^n$.

Equivalently:

**LEMMA** **(Invertibility)** A square matrix $A \in \mathbb{R}^{n \times n}$ is nonsingular if and only if there exists a unique $A^{-1}$ such that

$$
A A^{-1} = A^{-1} A = I_{n \times n}.
$$

The matrix $A^{-1}$ is referred to as the inverse of $A$. We also say that $A$ is invertible. $\flat$

*Proof idea:* We use the nonsingularity of $A$ to write the columns of the identity matrix as unique linear combinations of the columns of $A$.

*Proof:* Suppose first that $A$ has rank $n$. Then its columns are linearly independent and form a basis of $\mathbb{R}^n$. In particular, for any $i$ the standard basis vector $\mathbf{e}_i$ can be written as a unique linear combination of the columns of $A$, i.e., there is $\mathbf{b}_i$ such that $A \mathbf{b}_i =\mathbf{e}_i$. Let $B$ be the matrix with columns $\mathbf{b}_i$, $i=1,\ldots,n$. By construction, $A B = I_{n\times n}$. Applying the same idea to the rows of $A$ (which by the *Row Rank Equals Column Rank Lemma* also form a basis of $\mathbb{R}^n$), there is a unique $C$ such that $C A = I_{n\times n}$. Multiplying both sides by $B$, we get

$$
C = C A B = I_{n \times n} B = B.
$$

So we take $A^{-1} = B = C$.

In the other direction, following the same argument, the equation $A A^{-1} = I_{n \times n}$ implies that the standard basis of $\mathbb{R}^n$ is in the column space of $A$. So the columns of $A$ are a spanning list of all of $\mathbb{R}^n$ and $\mathrm{rk}(A) = n$. That proves the claim. $\square$

 **THEOREM** **(Inverting a Nonsingular System)** Let $A \in \mathbb{R}^{n \times n}$ be a nonsingular square matrix. Then for any $\mathbf{b} \in \mathbb{R}^n$, there exists a unique $\mathbf{x} \in \mathbb{R}^n$ such that $A \mathbf{x} = \mathbf{b}$. Moreover $\mathbf{x} = A^{-1} \mathbf{b}$. $\sharp$

*Proof:* The first claim follows immediately from the fact that the columns of $A$ form a basis of $\mathbb{R}^n$. For the second claim, note that 

$$
\mathbf{x} = A^{-1} A \mathbf{x} = A^{-1} \mathbf{b}.
$$ 

$\square$

**EXAMPLE:** Let $A \in \mathbb{R}^{n \times m}$ with $n \geq m$ have full column rank, i.e. rank $m$. We will show that the square matrix $B = A^T A$ is then invertible.

By a claim above, the columns of $A$ form a basis of its column space. In particular they are linearly independent. We will use this below. 

Observe that $B$ is an $m \times m$ matrix. By definition, to show that it is nonsingular, we need to establish that it has rank $m$, or put differently that its columns are also linearly independent. By the matrix version of the *Equivalent Definition of Linear Independence*, it suffices to show that 

$$
B \mathbf{x} = \mathbf{0} \implies \mathbf{x} = \mathbf{0}.
$$

We establish this next.

Since $B = A^T A$, the equation $B \mathbf{x} = \mathbf{0}$ implies

$$
A^T A \mathbf{x} = \mathbf{0}.
$$

Now comes the key idea: we multiply both sides by $\mathbf{x}^T$. The left-hand side becomes 

$$
\mathbf{x}^T (A^T A \mathbf{x}) 
= (A \mathbf{x})^T (A \mathbf{x}) 
= \|A \mathbf{x}\|^2,
$$

where we used that, for matrices $C, D$, we have $(CD)^T = D^T C^T$. The right-hand side becomes $\mathbf{x}^T \mathbf{0} = 0$. Hence we have shown that $A^T A \mathbf{x} = \mathbf{0}$ in fact implies $\|A \mathbf{x}\|^2 = 0$.

By the point-separating property of the Euclidean norm, the condition $\|A \mathbf{x}\|^2 = 0$ implies $A \mathbf{x} = \mathbf{0}$. Because $A$ has linearly independent columns, the *Equivalent Definition of Linear Independence* in its matrix form again implies that $\mathbf{x} = \mathbf{0}$, which is what we needed to prove. $\lhd$

**EXAMPLE:** Let $\mathbf{q}_1,\ldots,\mathbf{q}_n$ be an orthonormal basis of $\mathbb{R}^n$ and form the matrix 

$$
Q =
\begin{pmatrix}
| &  & | \\
\mathbf{q}_1 & \ldots & \mathbf{q}_n \\
| &  & | 
\end{pmatrix}.
$$

We show that $Q^{-1} = Q^T$. 

By the definition of matrix multiplication 

$$
Q^T Q
=
\begin{pmatrix}
\langle \mathbf{q}_1, \mathbf{q}_1 \rangle & \cdots & \langle \mathbf{q}_1, \mathbf{q}_n \rangle \\
\langle \mathbf{q}_2, \mathbf{q}_1 \rangle & \cdots & \langle \mathbf{q}_2, \mathbf{q}_n \rangle \\
\vdots & \ddots & \vdots \\
\langle \mathbf{q}_n, \mathbf{q}_1 \rangle & \cdots & \langle \mathbf{q}_n, \mathbf{q}_n \rangle
\end{pmatrix}
= I_{n \times n}
$$

where $I_{n \times n}$ denotes the $n \times n$ identity matrix. This of course follows from the fact that the $\mathbf{q}_i$'s are orthonormal. 

In the other direction, we claim that $Q Q^T = I_{n \times n}$ as well. Indeed the matrix $Q Q^T$ is the orthogonal projection on the span of the $\mathbf{q}_i$'s, that is, $\mathbb{R}^n$. By the *Orthogonal Projection Theorem*, the orthogonal projection $Q Q^T \mathbf{v}$ finds the closest vector to $\mathbf{v}$ in the span of the $\mathbf{q}_i$'s. But that span contains all vectors, including $\mathbf{v}$, so we must have $Q Q^T \mathbf{v} = \mathbf{v}$. Since this holds for all $\mathbf{v} \in \mathbb{R}^n$, the matrix $Q Q^T$ is the identity map and we have proved the claim. $\lhd$

Matrices that satisfy 

$$
Q^T Q = Q Q^T = I_{n \times n}
$$ 

are called orthogonal matrices.

**DEFINITION** **(Orthogonal Matrix)** A square matrix $Q \in \mathbb{R}^{m\times m}$ is orthogonal if $Q^T Q = Q Q^T = I_{m \times m}$. $\natural$